# 04B1 — Production HRRR Forecast Features

**Build ID: `04B1_HRRR_PRODUCTION_FEATURES_V2`**

This is the production HRRR notebook after the successful 04B0 pilot.

### Frozen rules

- Parent cohort: 04A_v3 units with `operational_anchor_clean=True`.
- Prediction anchor: earliest supported-event start from 04A_v2.
- Primary cycle: latest operationally available 00/06/12/18 UTC extended HRRR cycle.
- Conservative archive availability: initialization + 3 h must be no later than the prediction anchor.
- Missing-run fallback: try older extended cycles only when they were already available at the anchor and still cover the prediction anchor itself. Later fixed/adaptive horizons are populated only where that older forecast actually reaches; unavailable later horizons remain explicitly missing.
- Adaptive ETA: `exact_eta_history_median_h`, then `exact_eta_last_median_h`; query ETA clipped to 1–18 h.
- Units without ETA still receive all fixed-horizon weather features.
- No realized future weather is used.

04B0 showed that the pilot errors were concentrated in missing 2020 Zarr UGRD objects, while grouping reduced 304,143 clean units to only 3,057 incident × primary-run groups. Production therefore works by run and native Zarr spatial chunk rather than unit-by-unit remote requests.

### V2 correction

04B1 V2 contains two corrections discovered by the preflight:

1. approach-relative wind masking now uses broadcast-safe `np.where`;
2. HRRRv3 fallback requires operationally valid coverage through the prediction anchor, not artificially complete +18 h coverage. This salvages leakage-safe near-term weather from older valid cycles while leaving unsupported later horizons explicitly missing.

`hrrr_weather_available=True` means the selected run provides the required weather variables and reaches the prediction anchor. It does **not** imply every +18 h feature is available.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04B1_HRRR_PRODUCTION_FEATURES_V2"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
CORRECTED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_corrected_local_250m_10m"
UNIT_ROOT = CORRECTED_ROOT / "units"

EXACT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_extended_exact_approach_v3" /
    "extended_exact_approach_features_v3.parquet"
)

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "hrrr_production_v2"
SHARD_ROOT = PROCESSED_ROOT / "incident_run_shards"
OBJECT_CACHE_ROOT = PROJECT_ROOT / "data" / "raw" / "hrrr_cache" / "zarr_object_cache"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "hrrr_production_v2"

for p in [PROCESSED_ROOT, SHARD_ROOT, OBJECT_CACHE_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

assert EXACT_PATH.exists(), EXACT_PATH
assert UNIT_ROOT.exists(), UNIT_ROOT

CRS_METRIC = "EPSG:5070"
CRS_WGS84 = "EPSG:4326"

ZARR_AVAILABILITY_LAG_H = 3.0
EXTENDED_CYCLE_HOURS = {0, 6, 12, 18}
FIXED_HORIZONS_H = [0.0, 1.0, 3.0, 6.0, 12.0, 18.0]
FIXED_WINDOWS_H = [6, 12, 18]
ADAPTIVE_FRACTIONS = [0.25, 0.50, 0.75, 1.00]
ETA_MIN_H = 1.0
ETA_MAX_H = 18.0
MAX_EXTENDED_FALLBACK_STEPS = 4
USE_OBJECT_CACHE = False  # V2 default: avoid Zarr/fsspec async-cache warning; resumable shards still prevent repeated completed work
RESUME = True

HRRR_BUCKET = "hrrrzarr"
HRRR_VARIABLES = {
    "u10_grid_mps": ("10m_above_ground", "UGRD"),
    "v10_grid_mps": ("10m_above_ground", "VGRD"),
    "t2_k": ("2m_above_ground", "TMP"),
    "rh2_pct": ("2m_above_ground", "RH"),
    "gust_mps": ("surface", "GUST"),
}

print("RUNNING:", NOTEBOOK_BUILD_ID)
print("Input:", EXACT_PATH)
print("Output:", PROCESSED_ROOT)

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "pyarrow": "pyarrow",
    "shapely": "shapely",
    "tqdm": "tqdm",
    "s3fs": "s3fs",
    "fsspec": "fsspec",
    "xarray": "xarray",
    "zarr": "zarr",
    "pyproj": "pyproj",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import gc, math, time, traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import s3fs
import fsspec
import xarray as xr
import zarr

from pyproj import CRS, Transformer, Geod
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 300)

# HRRRv4 became operational at 12 UTC on 2 Dec 2020.
HRRR_V4_START_UTC = pd.Timestamp("2020-12-02T12:00:00Z")

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("zarr:", zarr.__version__)

## 1. Readers and ETA hierarchy

In [ ]:
def norm_irwin(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace("{", "", regex=False)
        .str.replace("}", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.lower()
        .replace({"<na>": pd.NA, "nan": pd.NA, "none": pd.NA, "": pd.NA})
    )


def read_03l_units(path):
    df = pd.read_parquet(path)
    geometry = gpd.GeoSeries.from_wkb(df["geometry"], crs=CRS_METRIC)
    return gpd.GeoDataFrame(
        df.drop(columns=["geometry"]),
        geometry=geometry,
        crs=CRS_METRIC,
    )


def resolve_unit_shard(raw_irwin):
    direct = UNIT_ROOT / f"{raw_irwin}.parquet"
    if direct.exists():
        return direct

    target = norm_irwin(pd.Series([raw_irwin])).iloc[0]
    matches = []

    for path in UNIT_ROOT.glob("*.parquet"):
        candidate = norm_irwin(pd.Series([path.stem])).iloc[0]
        if candidate == target:
            matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve unit shard for {raw_irwin}; matches={len(matches)}"
        )
    return matches[0]


exact = pd.read_parquet(EXACT_PATH)
assert exact["unit_uid"].is_unique

exact["prediction_anchor_utc"] = pd.to_datetime(
    exact["prediction_anchor_utc"], utc=True, errors="coerce"
)

exact["adaptive_eta_raw_h"] = exact["exact_eta_history_median_h"]
exact["adaptive_eta_source"] = np.where(
    exact["exact_eta_history_median_h"].notna(),
    "history_median",
    np.where(
        exact["exact_eta_last_median_h"].notna(),
        "last_median_fallback",
        "unavailable",
    ),
)

fallback = (
    exact["adaptive_eta_raw_h"].isna()
    & exact["exact_eta_last_median_h"].notna()
)

exact.loc[fallback, "adaptive_eta_raw_h"] = exact.loc[
    fallback, "exact_eta_last_median_h"
]

exact["adaptive_eta_query_h"] = exact["adaptive_eta_raw_h"].clip(
    lower=ETA_MIN_H,
    upper=ETA_MAX_H,
)

production = exact[
    exact["operational_anchor_clean"].fillna(False)
    & exact["prediction_anchor_utc"].notna()
].copy()

assert production["unit_uid"].is_unique

eta_summary = (
    production["adaptive_eta_source"]
    .value_counts(dropna=False)
    .rename("units")
    .reset_index()
)
eta_summary["pct"] = eta_summary["units"] / len(production) * 100

print("Clean production units:", f"{len(production):,}")
display(eta_summary)

## 2. Forecast-cycle rules

In [ ]:
def latest_available_extended_init(anchor_utc):
    cutoff = (
        pd.Timestamp(anchor_utc)
        - pd.Timedelta(hours=ZARR_AVAILABILITY_LAG_H)
    ).floor("h")

    candidate = cutoff
    while candidate.hour not in EXTENDED_CYCLE_HOURS:
        candidate -= pd.Timedelta(hours=1)

    return candidate


def previous_extended_cycle(init_utc, steps=1):
    return pd.Timestamp(init_utc) - pd.Timedelta(hours=6 * int(steps))


def nominal_extended_max_lead_h(init_utc):
    return 48 if pd.Timestamp(init_utc) >= HRRR_V4_START_UTC else 36


production["primary_extended_init_utc"] = production[
    "prediction_anchor_utc"
].map(latest_available_extended_init)

production["primary_forecast_age_h"] = (
    production["prediction_anchor_utc"]
    - production["primary_extended_init_utc"]
).dt.total_seconds() / 3600.0

print(
    "Unique primary extended runs:",
    production["primary_extended_init_utc"].nunique()
)
print(
    "Median age:",
    production["primary_forecast_age_h"].median(),
)

## 3. Unit centroids

In [ ]:
LOCATION_CACHE_PATH = PROCESSED_ROOT / "clean_unit_centroids_v1.parquet"

if LOCATION_CACHE_PATH.exists():
    locations = pd.read_parquet(LOCATION_CACHE_PATH)
else:
    frames = []

    for raw_irwin, grp in tqdm(
        production.groupby("_irwin_norm", sort=False),
        desc="Recover centroids",
    ):
        units = read_03l_units(resolve_unit_shard(raw_irwin))
        units = units[
            units["unit_uid"].isin(set(grp["unit_uid"]))
        ].copy()

        centroids = units.geometry.centroid

        loc = gpd.GeoDataFrame(
            units[["unit_uid"]].copy(),
            geometry=centroids,
            crs=CRS_METRIC,
        ).to_crs(CRS_WGS84)

        loc["longitude"] = loc.geometry.x
        loc["latitude"] = loc.geometry.y

        frames.append(loc[["unit_uid", "longitude", "latitude"]])

    locations = pd.concat(frames, ignore_index=True)
    assert locations["unit_uid"].is_unique
    locations.to_parquet(LOCATION_CACHE_PATH, index=False)

production = production.merge(
    locations,
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

if production[["longitude", "latitude"]].isna().any().any():
    raise RuntimeError("Missing production unit locations.")

print("Locations:", f"{len(locations):,}")

## 4. HRRR grid lookup and wind-grid rotation

In [ ]:
REMOTE_S3 = s3fs.S3FileSystem(anon=True)

GRID_INDEX_URL = "s3://hrrrzarr/grid/HRRR_chunk_index.zarr"

chunk_index = xr.open_zarr(
    s3fs.S3Map(GRID_INDEX_URL, s3=REMOTE_S3)
)

hrrr_x = np.asarray(chunk_index["x"].values, dtype=float)
hrrr_y = np.asarray(chunk_index["y"].values, dtype=float)

HRRR_CRS = CRS.from_proj4(
    "+proj=lcc +lat_1=38.5 +lat_2=38.5 +lat_0=38.5 "
    "+lon_0=-97.5 +a=6371229 +b=6371229 +units=m +no_defs"
)

TO_HRRR = Transformer.from_crs(CRS_WGS84, HRRR_CRS, always_xy=True)
FROM_HRRR = Transformer.from_crs(HRRR_CRS, CRS_WGS84, always_xy=True)
GEOD = Geod(a=6371229, b=6371229)


def forward_azimuth_deg(lon1, lat1, lon2, lat2):
    az, _, _ = GEOD.inv(lon1, lat1, lon2, lat2)
    return float(az % 360.0)


def grid_lookup_one(longitude, latitude):
    px, py = TO_HRRR.transform(longitude, latitude)

    ix1 = int(np.searchsorted(hrrr_x, px))
    iy1 = int(np.searchsorted(hrrr_y, py))

    ix1 = min(max(ix1, 1), len(hrrr_x) - 1)
    iy1 = min(max(iy1, 1), len(hrrr_y) - 1)

    ix0 = ix1 - 1
    iy0 = iy1 - 1

    wx = (px - hrrr_x[ix0]) / (hrrr_x[ix1] - hrrr_x[ix0])
    wy = (py - hrrr_y[iy0]) / (hrrr_y[iy1] - hrrr_y[iy0])

    lon_x, lat_x = FROM_HRRR.transform(px + 1000.0, py)
    lon_y, lat_y = FROM_HRRR.transform(px, py + 1000.0)

    az_x = forward_azimuth_deg(longitude, latitude, lon_x, lat_x)
    az_y = forward_azimuth_deg(longitude, latitude, lon_y, lat_y)

    return {
        "ix0": ix0, "ix1": ix1,
        "iy0": iy0, "iy1": iy1,
        "wx": float(wx), "wy": float(wy),
        "hrrr_grid_x_earth_azimuth_deg": az_x,
        "hrrr_grid_y_earth_azimuth_deg": az_y,
    }


GRID_CACHE_PATH = PROCESSED_ROOT / "clean_unit_hrrr_grid_lookup_v1.parquet"

if GRID_CACHE_PATH.exists():
    grid_lookup = pd.read_parquet(GRID_CACHE_PATH)
else:
    rows = []
    for rec in tqdm(
        production[["unit_uid", "longitude", "latitude"]].to_dict("records"),
        desc="Build HRRR grid lookup",
    ):
        rows.append({
            "unit_uid": rec["unit_uid"],
            **grid_lookup_one(rec["longitude"], rec["latitude"]),
        })

    grid_lookup = pd.DataFrame(rows)
    assert grid_lookup["unit_uid"].is_unique
    grid_lookup.to_parquet(GRID_CACHE_PATH, index=False)

production = production.merge(
    grid_lookup,
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

print("Grid lookup rows:", f"{len(grid_lookup):,}")

## 5. Persistent Zarr object cache and run validation

In [ ]:
OBJECT_CACHE_ACTIVE = False

if USE_OBJECT_CACHE:
    try:
        CACHED_FS = fsspec.filesystem(
            "filecache",
            target_protocol="s3",
            target_options={"anon": True},
            cache_storage=str(OBJECT_CACHE_ROOT),
            check_files=False,
        )
        OBJECT_CACHE_ACTIVE = True
        print("Persistent Zarr object cache: ACTIVE")
    except Exception as exc:
        print("Object-cache initialization failed; using direct S3.")
        print(type(exc).__name__, exc)


def hrrr_run_url(init_utc):
    init = pd.Timestamp(init_utc)
    date = init.strftime("%Y%m%d")
    hour = init.strftime("%H")
    return (
        f"s3://{HRRR_BUCKET}/sfc/{date}/"
        f"{date}_{hour}z_fcst.zarr"
    )


def make_run_mapper(init_utc):
    url = hrrr_run_url(init_utc)

    if OBJECT_CACHE_ACTIVE:
        return CACHED_FS.get_mapper(
            url.replace("s3://", "", 1)
        )

    return s3fs.S3Map(url, s3=REMOTE_S3)


RUN_STORE_CACHE = {}


def open_run_store(init_utc):
    key = pd.Timestamp(init_utc).isoformat()

    if key not in RUN_STORE_CACHE:
        RUN_STORE_CACHE[key] = zarr.open(
            make_run_mapper(init_utc),
            mode="r",
        )

    return RUN_STORE_CACHE[key]


def variable_key(level, variable):
    return f"{level}/{variable}/{level}/{variable}"


def validate_run_store(init_utc, required_max_lead_h):
    try:
        store = open_run_store(init_utc)

        lead_counts = []
        chunks = []

        for level, variable in HRRR_VARIABLES.values():
            key = variable_key(level, variable)

            if key not in store:
                return {
                    "ok": False,
                    "reason": f"missing_variable:{key}",
                }

            arr = store[key]

            if arr.ndim != 3:
                return {
                    "ok": False,
                    "reason": f"bad_shape:{key}:{arr.shape}",
                }

            lead_counts.append(int(arr.shape[0]))
            chunks.append(tuple(int(v) for v in arr.chunks))

        actual_leads = min(lead_counts)

        if required_max_lead_h > actual_leads:
            return {
                "ok": False,
                "reason": (
                    f"insufficient_actual_leads:"
                    f"{required_max_lead_h}>{actual_leads}"
                ),
                "actual_leads": actual_leads,
            }

        return {
            "ok": True,
            "reason": "OK",
            "actual_leads": actual_leads,
            "chunk_shape": chunks[0],
        }

    except Exception as exc:
        return {
            "ok": False,
            "reason": f"{type(exc).__name__}: {exc}",
        }


def resolve_group_cycle(group):
    """
    Select the latest leakage-safe extended HRRR cycle with all required
    variables and enough forecast lead to reach the prediction anchor for
    every unit in this incident × primary-run group.

    V2 deliberately does NOT require an older fallback cycle to reach
    anchor+18 h. HRRRv3 extended runs only reach F36, and requiring full
    +18 h coverage can throw away otherwise valid t0 / near-term weather.

    Later horizons are checked per unit inside process_group(). If an older
    fallback does not reach +12 or +18 h, those feature cells remain NaN.
    """
    primary = pd.Timestamp(
        group["primary_extended_init_utc"].iloc[0]
    )

    anchor_min = group["prediction_anchor_utc"].min()
    anchor_max = group["prediction_anchor_utc"].max()

    attempts = []

    for steps in range(MAX_EXTENDED_FALLBACK_STEPS + 1):
        candidate = previous_extended_cycle(primary, steps)

        # The run must have been available for the EARLIEST t0 in the group.
        available_by = (
            candidate
            + pd.Timedelta(hours=ZARR_AVAILABILITY_LAG_H)
        )

        if available_by > anchor_min:
            attempts.append({
                "candidate": candidate,
                "steps": steps,
                "status": "not_available_at_all_t0",
            })
            continue

        # Require the run to reach the LATEST prediction anchor in this group.
        required_t0_lead = int(
            math.ceil(
                (anchor_max - candidate).total_seconds() / 3600.0
            )
        )

        nominal_max = nominal_extended_max_lead_h(candidate)

        if required_t0_lead > nominal_max:
            attempts.append({
                "candidate": candidate,
                "steps": steps,
                "status": "cannot_reach_latest_t0_nominally",
                "required_t0_lead": required_t0_lead,
                "nominal_max": nominal_max,
            })
            continue

        check = validate_run_store(
            candidate,
            required_t0_lead,
        )

        actual_future_from_latest_t0_h = (
            check.get("actual_leads", np.nan)
            - (
                (anchor_max - candidate).total_seconds()
                / 3600.0
            )
            if check.get("ok", False)
            else np.nan
        )

        attempts.append({
            "candidate": candidate,
            "steps": steps,
            "status": check["reason"],
            "required_t0_lead": required_t0_lead,
            "nominal_max": nominal_max,
            "actual_future_from_latest_t0_h": actual_future_from_latest_t0_h,
        })

        if check["ok"]:
            return {
                "ok": True,
                "selected_init_utc": candidate,
                "fallback_steps": steps,
                "actual_store_leads": check["actual_leads"],
                "chunk_shape": check["chunk_shape"],
                "actual_future_from_latest_t0_h": actual_future_from_latest_t0_h,
                "attempts": attempts,
            }

    return {
        "ok": False,
        "selected_init_utc": pd.NaT,
        "fallback_steps": pd.NA,
        "actual_store_leads": pd.NA,
        "chunk_shape": None,
        "actual_future_from_latest_t0_h": np.nan,
        "attempts": attempts,
    }


## 6. Explicit 2020 fallback preflight

In [ ]:
KNOWN_PILOT_MISSING_RUN = pd.Timestamp("2020-10-20T18:00:00Z")

print("Known 04B0-missing run:")
display(pd.DataFrame([{
    "init": KNOWN_PILOT_MISSING_RUN,
    **validate_run_store(
        KNOWN_PILOT_MISSING_RUN,
        required_max_lead_h=24,
    ),
}]))

matching = production[
    production["primary_extended_init_utc"]
    == KNOWN_PILOT_MISSING_RUN
]

if len(matching):
    irwin = matching["_irwin_norm"].iloc[0]

    fallback_test_group = matching[
        matching["_irwin_norm"] == irwin
    ].copy()

    test_resolution = resolve_group_cycle(
        fallback_test_group
    )

    print("Fallback attempts:")
    display(pd.DataFrame(test_resolution["attempts"]))

    print(
        "Selected:",
        test_resolution["selected_init_utc"],
        "fallback steps:",
        test_resolution["fallback_steps"],
    )
else:
    print("No clean production group uses this primary run.")

## 7. Production manifest

In [ ]:
manifest = (
    production.groupby(
        ["_irwin_norm", "primary_extended_init_utc"],
        as_index=False,
    )
    .agg(
        IncidentName=("IncidentName", "first"),
        Year=("Year", "first"),
        units=("unit_uid", "size"),
        unique_anchor_times=("prediction_anchor_utc", "nunique"),
        anchor_min=("prediction_anchor_utc", "min"),
        anchor_max=("prediction_anchor_utc", "max"),
    )
)

manifest["group_id"] = [
    f"{irwin}__{pd.Timestamp(init).strftime('%Y%m%d_%H')}"
    for irwin, init
    in zip(
        manifest["_irwin_norm"],
        manifest["primary_extended_init_utc"],
    )
]

assert manifest["group_id"].is_unique

print("Production units:", f"{len(production):,}")
print("Groups:", f"{len(manifest):,}")
print("Median units/group:", manifest["units"].median())
print("P90:", manifest["units"].quantile(.90))
print("Max:", manifest["units"].max())

manifest.to_csv(
    OUTPUT_ROOT / "PRODUCTION_GROUP_MANIFEST.csv",
    index=False,
)

display(
    manifest.sort_values("units", ascending=False).head(20)
)

## 8. Cube interpolation and wind helpers

In [ ]:
def read_cube(store, level, variable, y0, y1, x0, x1):
    arr = store[variable_key(level, variable)]

    return np.asarray(
        arr[
            :,
            int(y0):int(y1) + 1,
            int(x0):int(x1) + 1,
        ],
        dtype=float,
    )


def interpolate_cube(cube, units, global_y0, global_x0):
    ly0 = units["iy0"].to_numpy(int) - int(global_y0)
    ly1 = units["iy1"].to_numpy(int) - int(global_y0)
    lx0 = units["ix0"].to_numpy(int) - int(global_x0)
    lx1 = units["ix1"].to_numpy(int) - int(global_x0)

    wx = units["wx"].to_numpy(float)[None, :]
    wy = units["wy"].to_numpy(float)[None, :]

    v00 = cube[:, ly0, lx0]
    v01 = cube[:, ly0, lx1]
    v10 = cube[:, ly1, lx0]
    v11 = cube[:, ly1, lx1]

    return (
        (1-wx)*(1-wy)*v00
        + wx*(1-wy)*v01
        + (1-wx)*wy*v10
        + wx*wy*v11
    )


def grid_uv_to_earth(u_grid, v_grid, az_x_deg, az_y_deg):
    ax = np.deg2rad(np.asarray(az_x_deg, dtype=float))
    ay = np.deg2rad(np.asarray(az_y_deg, dtype=float))

    east = u_grid*np.sin(ax) + v_grid*np.sin(ay)
    north = u_grid*np.cos(ax) + v_grid*np.cos(ay)

    return east, north


def relative_wind(east, north, approach_deg):
    """
    Rotate Earth east/north wind into components relative to the observed
    incoming fire-approach azimuth.

    Shapes:
        east, north : (forecast_hours, n_units)
        approach_deg: (1, n_units) or broadcast-compatible

    np.where is used deliberately because boolean assignment does not
    broadcast a (1, n_units) mask across forecast_hours.
    """
    a = np.deg2rad(
        np.asarray(
            approach_deg,
            dtype=float,
        )
    )

    ae = np.sin(a)
    an = np.cos(a)

    toward = (
        east * ae
        + north * an
    )

    cross = (
        -east * an
        + north * ae
    )

    valid = np.isfinite(a)

    # Broadcast-safe across the full forecast-time axis.
    toward = np.where(
        valid,
        toward,
        np.nan,
    ).astype(float, copy=False)

    cross = np.where(
        valid,
        cross,
        np.nan,
    ).astype(float, copy=False)

    return toward, cross


# Broadcast/missing-direction self-test.
_test_east = np.ones((36, 2), dtype=float)
_test_north = np.zeros((36, 2), dtype=float)
_test_approach = np.array([[90.0, np.nan]])

_test_toward, _test_cross = relative_wind(
    _test_east,
    _test_north,
    _test_approach,
)

assert _test_toward.shape == (36, 2)
assert np.isfinite(_test_toward[:, 0]).all()
assert np.isnan(_test_toward[:, 1]).all()

print("relative_wind broadcast self-test: PASS")


def rounded_lead(init_utc, valid_utc):
    exact_lead = (
        pd.Timestamp(valid_utc)
        - pd.Timestamp(init_utc)
    ).total_seconds() / 3600.0

    lead = int(np.round(exact_lead))

    selected_valid = (
        pd.Timestamp(init_utc)
        + pd.Timedelta(hours=lead)
    )

    error_min = (
        selected_valid
        - pd.Timestamp(valid_utc)
    ).total_seconds() / 60.0

    return lead, error_min


def window_leads(init_utc, anchor_utc, maximum_relative_h):
    rel = np.arange(
        0,
        int(math.floor(maximum_relative_h)) + 1,
        dtype=float,
    )

    if maximum_relative_h - math.floor(maximum_relative_h) > 1e-9:
        rel = np.append(rel, float(maximum_relative_h))

    leads = []

    for h in rel:
        valid = (
            pd.Timestamp(anchor_utc)
            + pd.Timedelta(hours=float(h))
        )
        lead, _ = rounded_lead(init_utc, valid)
        leads.append(lead)

    return np.unique(np.asarray(leads, dtype=int))


FIXED_LABELS = {
    0.0: "t0",
    1.0: "p1h",
    3.0: "p3h",
    6.0: "p6h",
    12.0: "p12h",
    18.0: "p18h",
}

ADAPTIVE_LABELS = {
    .25: "eta25",
    .50: "eta50",
    .75: "eta75",
    1.0: "eta100",
}

POINT_VARS = [
    "u10_grid_mps",
    "v10_grid_mps",
    "u10_earth_east_mps",
    "v10_earth_north_mps",
    "wind10_speed_mps",
    "gust_mps",
    "t2_c",
    "rh2_pct",
    "wind10_toward_approach_mps",
    "wind10_cross_approach_mps",
]

## 9. Per incident × primary-run extractor

In [ ]:
def make_empty_group(group, reason):
    out = group[
        [
            "unit_uid",
            "_irwin_norm",
            "IncidentName",
            "Year",
            "_engagement_norm",
            "hold_label",
            "prediction_anchor_utc",
            "primary_extended_init_utc",
            "adaptive_eta_source",
            "adaptive_eta_raw_h",
            "adaptive_eta_query_h",
            "approach_azimuth_deg",
            "approach_direction_reliable",
            "longitude",
            "latitude",
        ]
    ].copy()

    out["hrrr_weather_available"] = False
    out["hrrr_selected_init_utc"] = pd.NaT
    out["hrrr_fallback_steps"] = pd.NA
    out["hrrr_failure_reason"] = reason

    return out


def add_window_summaries(row, prefix, arrays, indices, j):
    if not len(indices):
        return

    t = arrays["t2_c"][indices, j]
    rh = arrays["rh2_pct"][indices, j]
    ws = arrays["wind10_speed_mps"][indices, j]
    gust = arrays["gust_mps"][indices, j]
    toward = arrays["wind10_toward_approach_mps"][indices, j]
    cross = arrays["wind10_cross_approach_mps"][indices, j]

    row[f"{prefix}_n_hours"] = len(indices)

    row[f"{prefix}_t2_c_mean"] = float(np.nanmean(t))
    row[f"{prefix}_t2_c_max"] = float(np.nanmax(t))

    row[f"{prefix}_rh2_pct_mean"] = float(np.nanmean(rh))
    row[f"{prefix}_rh2_pct_min"] = float(np.nanmin(rh))

    row[f"{prefix}_wind10_speed_mean_mps"] = float(np.nanmean(ws))
    row[f"{prefix}_wind10_speed_max_mps"] = float(np.nanmax(ws))

    row[f"{prefix}_gust_mean_mps"] = float(np.nanmean(gust))
    row[f"{prefix}_gust_max_mps"] = float(np.nanmax(gust))

    row[f"{prefix}_toward_mean_mps"] = float(np.nanmean(toward))
    row[f"{prefix}_toward_max_mps"] = float(np.nanmax(toward))

    row[f"{prefix}_cross_mean_mps"] = float(np.nanmean(cross))
    row[f"{prefix}_cross_abs_max_mps"] = float(
        np.nanmax(np.abs(cross))
    )


def process_group(group):
    group = group.copy()

    resolution = resolve_group_cycle(group)

    if not resolution["ok"]:
        attempts = " | ".join(
            f"{a.get('candidate')}:{a.get('status')}"
            for a in resolution["attempts"]
        )

        return (
            make_empty_group(
                group,
                "no_eligible_extended_cycle:" + attempts,
            ),
            {
                "ok": False,
                "failure_reason": "no_eligible_extended_cycle",
                "attempts": resolution["attempts"],
            },
        )

    selected_init = pd.Timestamp(
        resolution["selected_init_utc"]
    )

    store = open_run_store(selected_init)

    chunk_shape = tuple(resolution["chunk_shape"])
    _, chunk_y, chunk_x = chunk_shape

    group["_chunk_y"] = (
        group["iy0"].astype(int) // int(chunk_y)
    )
    group["_chunk_x"] = (
        group["ix0"].astype(int) // int(chunk_x)
    )

    out_frames = []
    variable_tile_reads = 0

    for (_, _), tile in group.groupby(
        ["_chunk_y", "_chunk_x"],
        sort=False,
    ):
        tile = tile.copy().reset_index(drop=True)

        y0 = int(tile["iy0"].min())
        y1 = int(tile["iy1"].max())
        x0 = int(tile["ix0"].min())
        x1 = int(tile["ix1"].max())

        raw = {}

        for name, (level, variable) in HRRR_VARIABLES.items():
            cube = read_cube(
                store,
                level,
                variable,
                y0,
                y1,
                x0,
                x1,
            )

            variable_tile_reads += 1

            raw[name] = interpolate_cube(
                cube,
                tile,
                y0,
                x0,
            )

        n_leads = min(v.shape[0] for v in raw.values())

        u_grid = raw["u10_grid_mps"][:n_leads, :]
        v_grid = raw["v10_grid_mps"][:n_leads, :]

        az_x = tile[
            "hrrr_grid_x_earth_azimuth_deg"
        ].to_numpy(float)[None, :]

        az_y = tile[
            "hrrr_grid_y_earth_azimuth_deg"
        ].to_numpy(float)[None, :]

        u_east, v_north = grid_uv_to_earth(
            u_grid,
            v_grid,
            az_x,
            az_y,
        )

        wind_speed = np.hypot(u_east, v_north)

        approach = tile[
            "approach_azimuth_deg"
        ].to_numpy(float)[None, :]

        toward, cross = relative_wind(
            u_east,
            v_north,
            approach,
        )

        arrays = {
            "u10_grid_mps": u_grid,
            "v10_grid_mps": v_grid,
            "u10_earth_east_mps": u_east,
            "v10_earth_north_mps": v_north,
            "wind10_speed_mps": wind_speed,
            "gust_mps": raw["gust_mps"][:n_leads, :],
            "t2_c": raw["t2_k"][:n_leads, :] - 273.15,
            "rh2_pct": raw["rh2_pct"][:n_leads, :],
            "wind10_toward_approach_mps": toward,
            "wind10_cross_approach_mps": cross,
        }

        rows = []

        for j, rec in enumerate(tile.to_dict("records")):
            row = {
                "unit_uid": rec["unit_uid"],
                "_irwin_norm": rec["_irwin_norm"],
                "IncidentName": rec["IncidentName"],
                "Year": rec["Year"],
                "_engagement_norm": rec["_engagement_norm"],
                "hold_label": rec["hold_label"],
                "prediction_anchor_utc": rec["prediction_anchor_utc"],
                "primary_extended_init_utc": rec["primary_extended_init_utc"],
                "hrrr_selected_init_utc": selected_init,
                "hrrr_fallback_steps": int(resolution["fallback_steps"]),
                "hrrr_actual_store_leads": int(resolution["actual_store_leads"]),
                "hrrr_weather_available": True,
                "hrrr_failure_reason": pd.NA,
                "adaptive_eta_source": rec["adaptive_eta_source"],
                "adaptive_eta_raw_h": rec["adaptive_eta_raw_h"],
                "adaptive_eta_query_h": rec["adaptive_eta_query_h"],
                "approach_azimuth_deg": rec["approach_azimuth_deg"],
                "approach_direction_reliable": rec["approach_direction_reliable"],
                "longitude": rec["longitude"],
                "latitude": rec["latitude"],
            }

            row["hrrr_forecast_age_at_t0_h"] = (
                pd.Timestamp(rec["prediction_anchor_utc"])
                - selected_init
            ).total_seconds() / 3600.0

            # Fixed point horizons.
            for rel_h in FIXED_HORIZONS_H:
                label = FIXED_LABELS[rel_h]
                valid = (
                    pd.Timestamp(rec["prediction_anchor_utc"])
                    + pd.Timedelta(hours=float(rel_h))
                )

                lead, error_min = rounded_lead(selected_init, valid)

                row[f"hrrr_{label}_lead_h"] = lead
                row[f"hrrr_{label}_valid_time_error_min"] = error_min

                if 1 <= lead <= n_leads:
                    idx = lead - 1
                    for name, series in arrays.items():
                        row[f"hrrr_{name}_{label}"] = float(series[idx, j])
                else:
                    for name in POINT_VARS:
                        row[f"hrrr_{name}_{label}"] = np.nan

            # Explicit fixed-horizon availability flags.
            # These are especially important for older HRRRv3 fallback cycles,
            # which may reach t0 and near-term horizons but not +18 h.
            row["hrrr_t0_available"] = bool(
                np.isfinite(row.get("hrrr_t2_c_t0", np.nan))
            )
            row["hrrr_p6h_available"] = bool(
                np.isfinite(row.get("hrrr_t2_c_p6h", np.nan))
            )
            row["hrrr_p12h_available"] = bool(
                np.isfinite(row.get("hrrr_t2_c_p12h", np.nan))
            )
            row["hrrr_p18h_available"] = bool(
                np.isfinite(row.get("hrrr_t2_c_p18h", np.nan))
            )

            # Adaptive point horizons.
            eta_q = rec.get("adaptive_eta_query_h", np.nan)

            for fraction in ADAPTIVE_FRACTIONS:
                label = ADAPTIVE_LABELS[fraction]

                if not np.isfinite(eta_q):
                    row[f"hrrr_{label}_lead_h"] = pd.NA
                    row[f"hrrr_{label}_relative_h"] = np.nan
                    row[f"hrrr_{label}_valid_time_error_min"] = np.nan

                    for name in POINT_VARS:
                        row[f"hrrr_{name}_{label}"] = np.nan

                    continue

                rel_h = float(eta_q) * float(fraction)

                valid = (
                    pd.Timestamp(rec["prediction_anchor_utc"])
                    + pd.Timedelta(hours=rel_h)
                )

                lead, error_min = rounded_lead(selected_init, valid)

                row[f"hrrr_{label}_lead_h"] = lead
                row[f"hrrr_{label}_relative_h"] = rel_h
                row[f"hrrr_{label}_valid_time_error_min"] = error_min

                if 1 <= lead <= n_leads:
                    idx = lead - 1
                    for name, series in arrays.items():
                        row[f"hrrr_{name}_{label}"] = float(series[idx, j])
                else:
                    for name in POINT_VARS:
                        row[f"hrrr_{name}_{label}"] = np.nan

            # Fixed hourly windows.
            for max_h in FIXED_WINDOWS_H:
                leads = window_leads(
                    selected_init,
                    rec["prediction_anchor_utc"],
                    float(max_h),
                )

                leads = leads[
                    (leads >= 1)
                    & (leads <= n_leads)
                ]

                indices = leads - 1

                add_window_summaries(
                    row,
                    f"hrrr_window_0_{max_h}h",
                    arrays,
                    indices,
                    j,
                )

            # Adaptive hourly window.
            if np.isfinite(eta_q):
                leads = window_leads(
                    selected_init,
                    rec["prediction_anchor_utc"],
                    float(eta_q),
                )

                leads = leads[
                    (leads >= 1)
                    & (leads <= n_leads)
                ]

                add_window_summaries(
                    row,
                    "hrrr_window_0_eta",
                    arrays,
                    leads - 1,
                    j,
                )

            rows.append(row)

        out_frames.append(pd.DataFrame(rows))

        del raw, arrays
        gc.collect()

    result = pd.concat(out_frames, ignore_index=True)

    assert result["unit_uid"].is_unique
    assert len(result) == len(group)

    metadata = {
        "ok": True,
        "selected_init_utc": selected_init,
        "fallback_steps": int(resolution["fallback_steps"]),
        "actual_store_leads": int(resolution["actual_store_leads"]),
        "actual_future_from_latest_t0_h": float(
            resolution.get("actual_future_from_latest_t0_h", np.nan)
        ),
        "chunk_shape": str(resolution["chunk_shape"]),
        "spatial_tile_groups": int(
            group.groupby(["_chunk_y", "_chunk_x"]).ngroups
        ),
        "variable_tile_reads": int(variable_tile_reads),
        "attempts": resolution["attempts"],
    }

    return result, metadata

## 10. Small-group production preflight

In [ ]:
manifest = (
    production.groupby(
        ["_irwin_norm", "primary_extended_init_utc"],
        as_index=False,
    )
    .agg(
        IncidentName=("IncidentName", "first"),
        Year=("Year", "first"),
        units=("unit_uid", "size"),
        unique_anchor_times=("prediction_anchor_utc", "nunique"),
        anchor_min=("prediction_anchor_utc", "min"),
        anchor_max=("prediction_anchor_utc", "max"),
    )
)

manifest["group_id"] = [
    f"{irwin}__{pd.Timestamp(init).strftime('%Y%m%d_%H')}"
    for irwin, init
    in zip(
        manifest["_irwin_norm"],
        manifest["primary_extended_init_utc"],
    )
]

assert manifest["group_id"].is_unique

manifest.to_csv(
    OUTPUT_ROOT / "PRODUCTION_GROUP_MANIFEST.csv",
    index=False,
)

candidate = manifest.sort_values("units").iloc[
    min(10, len(manifest)-1)
]

preflight_group = production[
    (production["_irwin_norm"] == candidate["_irwin_norm"])
    & (
        production["primary_extended_init_utc"]
        == pd.Timestamp(candidate["primary_extended_init_utc"])
    )
].copy()

print("Preflight group:", candidate["group_id"])
print("Units:", len(preflight_group))

start = time.perf_counter()

preflight_features, preflight_meta = process_group(
    preflight_group
)

runtime = time.perf_counter() - start

print("Runtime:", f"{runtime:.2f} s")

display(pd.DataFrame([{
    k: v
    for k, v in preflight_meta.items()
    if k != "attempts"
}]))

display(
    preflight_features[
        [
            "unit_uid",
            "prediction_anchor_utc",
            "hrrr_selected_init_utc",
            "hrrr_fallback_steps",
            "hrrr_weather_available",
            "hrrr_t2_c_t0",
            "hrrr_rh2_pct_t0",
            "hrrr_wind10_speed_mps_t0",
            "hrrr_wind10_toward_approach_mps_t0",
            "hrrr_t2_c_p18h",
        ]
    ].head(10)
)

assert len(preflight_features) == len(preflight_group)
assert preflight_features["unit_uid"].is_unique

print("04B1 SMALL-GROUP PREFLIGHT: PASS")

## 11. Resumable production

In [ ]:
RUN_LOG_PATH = OUTPUT_ROOT / "04B1_RUN_LOG.csv"

run_rows = []

bar = tqdm(
    manifest.to_dict("records"),
    total=len(manifest),
    desc="04B1 incident-run groups",
)

for m in bar:
    start = time.perf_counter()

    group_id = m["group_id"]
    raw_irwin = m["_irwin_norm"]
    primary_init = pd.Timestamp(m["primary_extended_init_utc"])

    shard_path = SHARD_ROOT / f"{group_id}.parquet"

    try:
        if RESUME and shard_path.exists():
            preview = pd.read_parquet(
                shard_path,
                columns=[
                    "unit_uid",
                    "hrrr_weather_available",
                    "hrrr_fallback_steps",
                ],
            )

            run_rows.append({
                "group_id": group_id,
                "_irwin_norm": raw_irwin,
                "primary_extended_init_utc": primary_init,
                "status": "SKIPPED_COMPLETE",
                "units": len(preview),
                "weather_available_units": int(
                    preview["hrrr_weather_available"].sum()
                ),
                "runtime_seconds": time.perf_counter() - start,
            })

            bar.set_postfix({
                "group": group_id[:18],
                "status": "resume",
            })
            continue

        group = production[
            (production["_irwin_norm"] == raw_irwin)
            & (production["primary_extended_init_utc"] == primary_init)
        ].copy()

        features, metadata = process_group(group)

        features.to_parquet(
            shard_path,
            index=False,
        )

        runtime = time.perf_counter() - start

        run_row = {
            "group_id": group_id,
            "_irwin_norm": raw_irwin,
            "primary_extended_init_utc": primary_init,
            "status": "OK" if metadata["ok"] else "NO_WEATHER",
            "units": len(features),
            "weather_available_units": int(
                features["hrrr_weather_available"].sum()
            ),
            "runtime_seconds": runtime,
        }

        for key in [
            "selected_init_utc",
            "fallback_steps",
            "actual_store_leads",
            "actual_future_from_latest_t0_h",
            "chunk_shape",
            "spatial_tile_groups",
            "variable_tile_reads",
            "failure_reason",
        ]:
            if key in metadata:
                run_row[key] = metadata[key]

        run_rows.append(run_row)

        bar.set_postfix({
            "group": group_id[:18],
            "units": len(features),
            "available": int(
                features["hrrr_weather_available"].sum()
            ),
            "fallback": metadata.get("fallback_steps", "-"),
            "sec": f"{runtime:.1f}",
        })

        del features
        gc.collect()

    except Exception as exc:
        (OUTPUT_ROOT / f"ERROR_{group_id}.txt").write_text(
            traceback.format_exc(),
            encoding="utf-8",
        )

        run_rows.append({
            "group_id": group_id,
            "_irwin_norm": raw_irwin,
            "primary_extended_init_utc": primary_init,
            "status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
            "runtime_seconds": time.perf_counter() - start,
        })

        bar.set_postfix({
            "group": group_id[:18],
            "status": "ERROR",
        })

    pd.DataFrame(run_rows).to_csv(
        RUN_LOG_PATH,
        index=False,
    )

## 12. Completion check and combine

In [ ]:
shards = sorted(SHARD_ROOT.glob("*.parquet"))
expected = len(manifest)

print("Completed shards:", len(shards), "/", expected)

if len(shards) != expected:
    log = pd.read_csv(RUN_LOG_PATH)

    display(
        log[log["status"] == "ERROR"]
    )

    raise RuntimeError(
        "04B1 incomplete. Fix true errors and rerun production; "
        "completed shards will be reused."
    )

hrrr_features = pd.concat(
    [
        pd.read_parquet(path)
        for path in tqdm(
            shards,
            desc="Combine HRRR shards",
        )
    ],
    ignore_index=True,
)

assert hrrr_features["unit_uid"].is_unique
assert len(hrrr_features) == len(production)

FEATURES_PATH = (
    PROCESSED_ROOT /
    "hrrr_operational_forecast_features_v1.parquet"
)

hrrr_features.to_parquet(
    FEATURES_PATH,
    index=False,
)

print("Rows:", f"{len(hrrr_features):,}")
print("Columns:", len(hrrr_features.columns))
print("Saved:", FEATURES_PATH)

## 13. Production QA

In [ ]:
cycle_audit = (
    hrrr_features.groupby(
        ["hrrr_weather_available", "hrrr_fallback_steps"],
        dropna=False,
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        selected_runs=("hrrr_selected_init_utc", "nunique"),
    )
)

cycle_audit["pct_units"] = (
    cycle_audit["units"]
    / len(hrrr_features)
    * 100
)

display(cycle_audit)

availability = (
    hrrr_features.groupby(
        "_engagement_norm",
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        weather_available=("hrrr_weather_available", "sum"),
    )
)

availability["weather_available_pct"] = (
    availability["weather_available"]
    / availability["units"]
    * 100
)

display(availability)

error_cols = [
    c for c in hrrr_features.columns
    if c.endswith("_valid_time_error_min")
]

rounding = []

for c in error_cols:
    x = hrrr_features[c].dropna().abs()

    rounding.append({
        "feature": c,
        "available": len(x),
        "median_abs_error_min": x.median() if len(x) else np.nan,
        "p90_abs_error_min": x.quantile(.90) if len(x) else np.nan,
        "max_abs_error_min": x.max() if len(x) else np.nan,
    })

rounding_audit = pd.DataFrame(rounding)

display(rounding_audit)

grid_speed = np.hypot(
    hrrr_features["hrrr_u10_grid_mps_t0"],
    hrrr_features["hrrr_v10_grid_mps_t0"],
)

earth_speed = np.hypot(
    hrrr_features["hrrr_u10_earth_east_mps_t0"],
    hrrr_features["hrrr_v10_earth_north_mps_t0"],
)

rotation_error = np.abs(
    grid_speed - earth_speed
).dropna()

rotation_qa = pd.DataFrame([{
    "n": len(rotation_error),
    "median_abs_error_mps": rotation_error.median(),
    "p99_abs_error_mps": rotation_error.quantile(.99),
    "max_abs_error_mps": rotation_error.max(),
}])

display(rotation_qa)

plausibility_vars = [
    "hrrr_t2_c_t0",
    "hrrr_rh2_pct_t0",
    "hrrr_wind10_speed_mps_t0",
    "hrrr_gust_mps_t0",
    "hrrr_wind10_toward_approach_mps_t0",
    "hrrr_wind10_cross_approach_mps_t0",
    "hrrr_t2_c_p18h",
    "hrrr_rh2_pct_p18h",
    "hrrr_wind10_speed_mps_p18h",
    "hrrr_gust_mps_p18h",
]

plausibility = []

for c in plausibility_vars:
    x = hrrr_features[c].dropna()

    plausibility.append({
        "variable": c,
        "n": len(x),
        "min": x.min() if len(x) else np.nan,
        "p01": x.quantile(.01) if len(x) else np.nan,
        "median": x.median() if len(x) else np.nan,
        "p99": x.quantile(.99) if len(x) else np.nan,
        "max": x.max() if len(x) else np.nan,
    })

plausibility = pd.DataFrame(plausibility)
display(plausibility)

cycle_audit.to_csv(
    OUTPUT_ROOT / "FORECAST_CYCLE_FALLBACK_AUDIT.csv",
    index=False,
)
availability.to_csv(
    OUTPUT_ROOT / "WEATHER_AVAILABILITY_BY_CLASS.csv",
    index=False,
)
rounding_audit.to_csv(
    OUTPUT_ROOT / "VALID_TIME_ROUNDING_AUDIT.csv",
    index=False,
)
rotation_qa.to_csv(
    OUTPUT_ROOT / "WIND_ROTATION_QA.csv",
    index=False,
)
plausibility.to_csv(
    OUTPUT_ROOT / "PRODUCTION_WEATHER_PLAUSIBILITY.csv",
    index=False,
)

## 14. Missingness, runtime, and final report

In [ ]:
weather_cols = [
    c for c in hrrr_features.columns
    if c.startswith("hrrr_")
]

missingness = pd.DataFrame({
    "feature": weather_cols,
    "missing_count": [
        hrrr_features[c].isna().sum()
        for c in weather_cols
    ],
})

missingness["missing_pct"] = (
    missingness["missing_count"]
    / len(hrrr_features)
    * 100
)

missingness = missingness.sort_values(
    ["missing_pct", "feature"],
    ascending=[False, True],
)

display(missingness.head(100))

missingness.to_csv(
    OUTPUT_ROOT / "HRRR_FEATURE_MISSINGNESS.csv",
    index=False,
)

run_log = pd.read_csv(RUN_LOG_PATH)

performance = (
    run_log.groupby("status", as_index=False)
    .agg(
        groups=("group_id", "size"),
        units=("units", "sum"),
        total_runtime_s=("runtime_seconds", "sum"),
        median_group_runtime_s=("runtime_seconds", "median"),
        p90_group_runtime_s=(
            "runtime_seconds",
            lambda s: s.quantile(.90),
        ),
        max_group_runtime_s=("runtime_seconds", "max"),
    )
)

display(performance)

performance.to_csv(
    OUTPUT_ROOT / "PRODUCTION_RUNTIME_AUDIT.csv",
    index=False,
)


def directory_size_gb(root):
    total = 0

    for p in Path(root).rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass

    return total / (1024**3)


cache_size_gb = directory_size_gb(
    OBJECT_CACHE_ROOT
)


def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


report = f"""# 04B1 Production HRRR Forecast Feature Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Production cohort

Clean operational-anchor units: **{len(production):,}**

Output rows: **{len(hrrr_features):,}**

Output columns: **{len(hrrr_features.columns):,}**

## Forecast rule

Primary cycle: latest operationally available 00/06/12/18 UTC extended HRRR cycle.

Availability rule:

`initialization + {ZARR_AVAILABILITY_LAG_H:g} h <= prediction anchor`.

Older extended cycles are used only when they were already available and reach the prediction anchor for every unit in the group. If an older HRRRv3 fallback does not reach a later +12/+18 h horizon, that later feature remains explicitly missing rather than discarding valid t0 weather.

Realized future weather is never used.

## ETA hierarchy

{tomd(eta_summary)}

Primary: `exact_eta_history_median_h`

Fallback: `exact_eta_last_median_h`

Query ETA clipped to [{ETA_MIN_H:g}, {ETA_MAX_H:g}] h.

Units with unavailable ETA retain fixed weather features and have adaptive features missing by design.

## Forecast fallback audit

{tomd(cycle_audit)}

## Weather availability

{tomd(availability)}

## Valid-time rounding

{tomd(rounding_audit)}

## Wind-rotation QA

{tomd(rotation_qa)}

## Weather plausibility

{tomd(plausibility)}

## Runtime

{tomd(performance)}

Persistent Zarr-object cache size: **{cache_size_gb:.2f} GB**

## Feature semantics

HRRR remains a ~3-km atmospheric product. Bilinear interpolation to a fireline-unit centroid does not create 250-m meteorological resolution.

Fixed point features are generated at 0, 1, 3, 6, 12, and 18 h relative to the operational prediction anchor.

Fixed hourly window summaries are generated over 0–6, 0–12, and 0–18 h.

When ETA exists, adaptive point features are generated at 0.25τ, 0.50τ, 0.75τ, and τ, together with an hourly 0–τ summary.

Raw grid-relative wind, rotated Earth east/north wind, wind speed, and approach-relative wind are retained.

Rows are never silently dropped because of incomplete HRRR-Zarr runs. `hrrr_weather_available=True` means the chosen operational run reaches t0 with all required variables; per-horizon availability flags distinguish whether +6, +12, and +18 h are also present. If no leakage-safe cycle reaches t0, the row remains with explicit weather-unavailable flags.

## Next step

If these production audits are acceptable, freeze HRRR V1 and proceed to:

`04C_terrain_features.ipynb`.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "HRRR_PRODUCTION_FEATURE_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)

## 15. Output inventory

In [ ]:
print("=" * 96)
print("04B1 OUTPUTS")
print("=" * 96)

print("Primary feature table:")
print(FEATURES_PATH)

print()
print("Report:")
print(REPORT_PATH)

print()
print("Output files:")

for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file():
        print(
            f"{p.name:<70}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "Shard count:",
    len(list(SHARD_ROOT.glob("*.parquet")))
)

print()
print(
    "Persistent object cache:",
    f"{cache_size_gb:.2f} GB"
)

print()
print("DONE:", NOTEBOOK_BUILD_ID)

In [ ]:
# ============================================================
# 04B1_v2 — HRRR MISSING-SENTINEL AUDIT
# Run AFTER hrrr_features has been combined
# ============================================================

print("=" * 95)
print("A. GUST SENTINEL AUDIT")
print("=" * 95)

gust_columns = [
    c for c in hrrr_features.columns
    if "gust" in c.lower()
]

print("Gust-related columns:", len(gust_columns))

audit_rows = []

for c in gust_columns:

    x = pd.to_numeric(
        hrrr_features[c],
        errors="coerce",
    )

    # Physical gust speed cannot be negative.
    negative = x < 0

    # Specifically track obvious encoded missing values.
    sentinel = x <= -1000

    audit_rows.append({
        "feature": c,
        "available": int(x.notna().sum()),
        "negative_count": int(negative.sum()),
        "negative_pct": float(
            negative.mean() * 100
        ),
        "sentinel_le_minus1000_count": int(
            sentinel.sum()
        ),
        "sentinel_le_minus1000_pct": float(
            sentinel.mean() * 100
        ),
        "minimum": float(
            x.min()
        ) if x.notna().any() else np.nan,
        "p001": float(
            x.quantile(0.001)
        ) if x.notna().any() else np.nan,
    })

gust_audit = (
    pd.DataFrame(audit_rows)
    .sort_values(
        [
            "sentinel_le_minus1000_count",
            "negative_count",
        ],
        ascending=False,
    )
)

display(gust_audit)


print()
print("=" * 95)
print("B. AFFECTED UNITS")
print("=" * 95)

gust_point_columns = [
    c for c in gust_columns
    if (
        "_gust_mps_" in c
        and "window" not in c
    )
]

gust_mean_columns = [
    c for c in gust_columns
    if "gust_mean_mps" in c
]

gust_max_columns = [
    c for c in gust_columns
    if "gust_max_mps" in c
]

point_bad = pd.Series(
    False,
    index=hrrr_features.index,
)

for c in gust_point_columns:
    point_bad |= (
        pd.to_numeric(
            hrrr_features[c],
            errors="coerce",
        )
        < 0
    )

window_mean_bad = pd.Series(
    False,
    index=hrrr_features.index,
)

for c in gust_mean_columns:
    window_mean_bad |= (
        pd.to_numeric(
            hrrr_features[c],
            errors="coerce",
        )
        < 0
    )

window_max_bad = pd.Series(
    False,
    index=hrrr_features.index,
)

for c in gust_max_columns:
    window_max_bad |= (
        pd.to_numeric(
            hrrr_features[c],
            errors="coerce",
        )
        < 0
    )

any_bad = (
    point_bad
    | window_mean_bad
    | window_max_bad
)

print(
    "Units with bad point gust:",
    f"{int(point_bad.sum()):,}",
    f"({point_bad.mean()*100:.4f}%)",
)

print(
    "Units with bad window-mean gust:",
    f"{int(window_mean_bad.sum()):,}",
    f"({window_mean_bad.mean()*100:.4f}%)",
)

print(
    "Units with bad window-max gust:",
    f"{int(window_max_bad.sum()):,}",
    f"({window_max_bad.mean()*100:.4f}%)",
)

print(
    "Units affected anywhere:",
    f"{int(any_bad.sum()):,}",
    f"({any_bad.mean()*100:.4f}%)",
)


print()
print("=" * 95)
print("C. INCIDENT / RUN CONCENTRATION")
print("=" * 95)

affected = hrrr_features.loc[
    any_bad,
    [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "hrrr_selected_init_utc",
        "hrrr_fallback_steps",
    ],
].copy()

if len(affected):

    affected_runs = (
        affected.groupby(
            [
                "Year",
                "_irwin_norm",
                "IncidentName",
                "hrrr_selected_init_utc",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            affected_units=(
                "unit_uid",
                "size",
            )
        )
        .sort_values(
            "affected_units",
            ascending=False,
        )
    )

    print(
        "Affected incidents:",
        affected["_irwin_norm"].nunique()
    )

    print(
        "Affected selected HRRR runs:",
        affected[
            "hrrr_selected_init_utc"
        ].nunique()
    )

    display(
        affected_runs.head(50)
    )

else:

    print("No affected rows.")


print()
print("=" * 95)
print("D. OTHER OBVIOUS NUMERIC SENTINELS")
print("=" * 95)

# Search HRRR floating-point feature columns for extremely negative values.
candidate_numeric = [
    c for c in hrrr_features.columns
    if (
        c.startswith("hrrr_")
        and pd.api.types.is_numeric_dtype(
            hrrr_features[c]
        )
    )
]

extreme_rows = []

for c in candidate_numeric:

    x = pd.to_numeric(
        hrrr_features[c],
        errors="coerce",
    )

    n_extreme = int(
        (x <= -1000).sum()
    )

    if n_extreme:

        extreme_rows.append({
            "feature": c,
            "count_le_minus1000": n_extreme,
            "pct": n_extreme / len(x) * 100,
            "minimum": x.min(),
        })

extreme_audit = (
    pd.DataFrame(extreme_rows)
    .sort_values(
        "count_le_minus1000",
        ascending=False,
    )
    if extreme_rows
    else pd.DataFrame(
        columns=[
            "feature",
            "count_le_minus1000",
            "pct",
            "minimum",
        ]
    )
)

display(extreme_audit)

In [ ]:
# ============================================================
# 04B1_v2 FINAL GUST-SENTINEL CLEANUP
#
# Purpose:
#   Replace invalid HRRR gust missing-value sentinels with NaN
#   WITHOUT rerunning HRRR extraction.
#
# Scientific rule:
#   Gust speed cannot be negative.
#   The diagnosed invalid values are -10000.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 95)
print("04B1_v2 FINAL GUST-SENTINEL CLEANUP")
print("=" * 95)


# ------------------------------------------------------------
# 1. Work on a copy of the completed production table
# ------------------------------------------------------------

hrrr_clean = hrrr_features.copy()


# ------------------------------------------------------------
# 2. Identify every gust-derived numeric feature
# ------------------------------------------------------------

gust_columns = [
    c
    for c in hrrr_clean.columns
    if "gust" in c.lower()
    and pd.api.types.is_numeric_dtype(
        hrrr_clean[c]
    )
]

print(
    "Gust-related numeric columns:",
    len(gust_columns)
)


# ------------------------------------------------------------
# 3. Identify invalid gust sentinels BEFORE modification
# ------------------------------------------------------------

bad_by_column = {}

for c in gust_columns:

    values = pd.to_numeric(
        hrrr_clean[c],
        errors="coerce",
    )

    # Physical gust speed is nonnegative.
    # In this production audit every negative value diagnosed
    # was the HRRR missing sentinel -10000.
    bad = values < 0

    bad_by_column[c] = int(
        bad.sum()
    )


bad_any = pd.Series(
    False,
    index=hrrr_clean.index,
)

for c in gust_columns:

    values = pd.to_numeric(
        hrrr_clean[c],
        errors="coerce",
    )

    bad_any |= (
        values < 0
    )


print(
    "Affected units before cleanup:",
    f"{int(bad_any.sum()):,}",
    f"({bad_any.mean()*100:.6f}%)",
)


# ------------------------------------------------------------
# 4. Preserve explicit QA provenance
# ------------------------------------------------------------

hrrr_clean[
    "hrrr_gust_sentinel_cleaned_any"
] = bad_any

hrrr_clean[
    "hrrr_gust_sentinel_cleaned_count"
] = 0


for c in gust_columns:

    values = pd.to_numeric(
        hrrr_clean[c],
        errors="coerce",
    )

    bad = values < 0

    hrrr_clean.loc[
        bad,
        "hrrr_gust_sentinel_cleaned_count"
    ] += 1

    hrrr_clean.loc[
        bad,
        c
    ] = np.nan


# ------------------------------------------------------------
# 5. Validate that no negative gust survives
# ------------------------------------------------------------

remaining_negative = {}

for c in gust_columns:

    values = pd.to_numeric(
        hrrr_clean[c],
        errors="coerce",
    )

    n = int(
        (
            values < 0
        ).sum()
    )

    if n:
        remaining_negative[c] = n


assert not remaining_negative, (
    "Negative gust values remain: "
    f"{remaining_negative}"
)


# ------------------------------------------------------------
# 6. Verify that no other extreme HRRR sentinel remains
# ------------------------------------------------------------

extreme_remaining = []

for c in hrrr_clean.columns:

    if (
        c.startswith("hrrr_")
        and pd.api.types.is_numeric_dtype(
            hrrr_clean[c]
        )
    ):

        values = pd.to_numeric(
            hrrr_clean[c],
            errors="coerce",
        )

        n = int(
            (
                values <= -1000
            ).sum()
        )

        if n:

            extreme_remaining.append(
                {
                    "feature": c,
                    "count_le_minus1000": n,
                    "minimum": values.min(),
                }
            )


extreme_remaining = pd.DataFrame(
    extreme_remaining
)

print()
print(
    "Remaining HRRR values <= -1000:",
    len(extreme_remaining)
)

if len(extreme_remaining):

    display(
        extreme_remaining
    )

else:

    print(
        "PASS: no obvious <= -1000 HRRR sentinel remains."
    )


# ------------------------------------------------------------
# 7. Post-cleanup gust QA
# ------------------------------------------------------------

qa_rows = []

for c in gust_columns:

    x = pd.to_numeric(
        hrrr_clean[c],
        errors="coerce",
    )

    qa_rows.append(
        {
            "feature": c,

            "cleaned_values": (
                bad_by_column[c]
            ),

            "available_after_cleanup": int(
                x.notna().sum()
            ),

            "negative_after_cleanup": int(
                (
                    x < 0
                ).sum()
            ),

            "minimum_after_cleanup": (
                float(
                    x.min()
                )
                if x.notna().any()
                else np.nan
            ),

            "p001_after_cleanup": (
                float(
                    x.quantile(
                        0.001
                    )
                )
                if x.notna().any()
                else np.nan
            ),
        }
    )


gust_cleanup_audit = pd.DataFrame(
    qa_rows
)

display(
    gust_cleanup_audit
)


# ------------------------------------------------------------
# 8. Class balance of affected units
# ------------------------------------------------------------

affected_summary = (
    hrrr_clean[
        hrrr_clean[
            "hrrr_gust_sentinel_cleaned_any"
        ]
    ]
    .groupby(
        "_engagement_norm",
        as_index=False,
    )
    .agg(
        affected_units=(
            "unit_uid",
            "size",
        ),

        incidents=(
            "_irwin_norm",
            "nunique",
        ),
    )
)

display(
    affected_summary
)


# ------------------------------------------------------------
# 9. Save CLEAN production table
#
# Do not overwrite the original raw production table.
# This gives us provenance:
#
#   original extracted table
#   +
#   explicitly cleaned modeling table
# ------------------------------------------------------------

CLEAN_FEATURES_PATH = (
    PROCESSED_ROOT
    / "hrrr_operational_forecast_features_v2_clean.parquet"
)

hrrr_clean.to_parquet(
    CLEAN_FEATURES_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Save cleanup audit
# ------------------------------------------------------------

GUST_CLEANUP_AUDIT_PATH = (
    OUTPUT_ROOT
    / "HRRR_GUST_SENTINEL_CLEANUP_AUDIT.csv"
)

gust_cleanup_audit.to_csv(
    GUST_CLEANUP_AUDIT_PATH,
    index=False,
)


affected_summary.to_csv(
    OUTPUT_ROOT
    / "HRRR_GUST_SENTINEL_AFFECTED_CLASS_SUMMARY.csv",
    index=False,
)


# ------------------------------------------------------------
# 11. Final assertions
# ------------------------------------------------------------

assert len(
    hrrr_clean
) == len(
    hrrr_features
)

assert hrrr_clean[
    "unit_uid"
].is_unique

assert (
    hrrr_clean[
        "unit_uid"
    ].tolist()
    == hrrr_features[
        "unit_uid"
    ].tolist()
)


print()
print("=" * 95)
print("CLEANUP COMPLETE")
print("=" * 95)

print(
    "Rows:",
    f"{len(hrrr_clean):,}"
)

print(
    "Affected units:",
    f"{int(hrrr_clean['hrrr_gust_sentinel_cleaned_any'].sum()):,}"
)

print(
    "Affected percentage:",
    f"{hrrr_clean['hrrr_gust_sentinel_cleaned_any'].mean()*100:.6f}%"
)

print()
print(
    "Clean HRRR feature table:"
)

print(
    CLEAN_FEATURES_PATH
)

print()
print(
    "Cleanup audit:"
)

print(
    GUST_CLEANUP_AUDIT_PATH
)

print()
print(
    "HRRR GUST SENTINEL CLEANUP: PASS"
)